# NB_lab_test_trends
Builds `<catalog>.gold.lab_test_trends` per STTM *Silver to Gold (CareSync Health Network) v1.0*.

* **Grain:** one row per test type (test_name) per calendar month
* **Driving table:** `silver.lab_results` grouped by test_name, year(test_date), month(test_date)
* **Joins:** LEFT JOIN `silver.calendar` on `datekey = int(date_format(first day of month, 'yyyyMMdd'))` for month_name / quarter_name
* **Window:** LAG(abnormal_rate_pct) partitioned by test_name ordered by year, month
* **Filters:** none
* **Load:** FULL rebuild - CREATE OR REPLACE TABLE ... AS SELECT

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE databeli_adb.gold.lab_test_trends
COMMENT 'Gold: one row per test_name per month with abnormal/critical rates and month-over-month trend. Full rebuild from silver.'
AS
WITH monthly AS (
  SELECT test_name,
         YEAR(test_date)                      AS year,
         MONTH(test_date)                     AS month,
         COUNT(*)                             AS total_tests,
         COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
         COUNT_IF(result_status = 'Critical') AS critical_test_count
  FROM databeli_adb.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT m.*,
         CAST(ROUND(abnormal_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
         CAST(ROUND(critical_test_count * 100.0 / NULLIF(total_tests, 0), 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT r.*,
         LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  CAST(w.test_name AS STRING)            AS test_name,
  CAST(w.year AS INT)                    AS year,
  CAST(w.month AS INT)                   AS month,
  CAST(c.month_name AS STRING)           AS month_name,
  CAST(c.quarter_name AS STRING)         AS quarter_name,
  CAST(w.total_tests AS INT)             AS total_tests,
  CAST(w.abnormal_test_count AS INT)     AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)     AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                       AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN databeli_adb.silver.calendar c
       ON c.datekey = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)
""")

In [0]:
updated_df = spark.read.table("databeli_adb.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))